# optimizer-state-tensor-buffers — worked example 3: In-place buffer update preserves data pointer

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-state-tensor-buffers`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

State buffers in optimizers must be updated in-place using methods like `.copy_()`, `.mul_()`, or `.add_()`. If you assign a new tensor (`b = momentum * b + grad`), the optimizer's internal list still points to the old buffer — the new tensor is an orphan and the state is lost. In-place operations keep the same storage address (data pointer) across steps.

## Worked solution

**Step 1 — Allocate a buffer with `zeros_like`.**
After allocation, we record the `data_ptr()` — the memory address of the buffer's storage.

**Step 2 — Perform an in-place update.**
We use `buf.copy_(new_value)` to update the buffer. After the call, `buf.data_ptr()` is the same as before — same storage, new values.

**Step 3 — Contrast with out-of-place assignment.**
If we instead write `buf = new_value` (assignment), `buf` now points to a fresh tensor with a different `data_ptr`. The original buffer in the list is unchanged.

**Step 4 — Demonstrate why this matters.**
We show a buggy optimizer (out-of-place assignment) and a correct one (in-place). After 3 steps, the buggy optimizer's `self.bufs` list still holds the original all-zero buffers.

In [ ]:
import torch as t

class Momentum:
    """SGD with momentum. Buffer updated IN-PLACE so the same tensor
    (same storage / data_ptr) persists across every step."""
    def __init__(self, params, lr, momentum=0.9):
        self.params   = list(params)
        self.lr       = lr
        self.momentum = momentum
        self.bufs     = [t.zeros_like(p) for p in self.params]

    @t.no_grad()
    def step(self):
        for p, buf in zip(self.params, self.bufs):
            if p.grad is None:
                continue
            # in-place: keep the SAME buffer tensor, accumulate momentum + grad
            buf.mul_(self.momentum).add_(p.grad)
            p -= self.lr * buf

    def zero_grad(self):
        for p in self.params:
            p.grad = None

# --- demonstrate buffer persistence ---
t.manual_seed(0)
p = t.tensor([5.0], requires_grad=True)
opt = Momentum([p], lr=0.1)

orig_ptr = opt.bufs[0].data_ptr()

for _ in range(3):
    p.grad = t.tensor([1.0])
    opt.step()
    opt.zero_grad()

new_ptr = opt.bufs[0].data_ptr()

# expected accumulation: buf_0=0 -> 1 -> 1.9 -> 2.71
expected = 0.0
for _ in range(3):
    expected = 0.9 * expected + 1.0
print(f'buffer value: {opt.bufs[0].item():.4f}  (expected {expected:.4f})')
print(f'buffer ptr preserved across steps: {orig_ptr == new_ptr}')

assert orig_ptr == new_ptr, 'in-place update must preserve the same buffer storage'
assert abs(opt.bufs[0].item() - expected) < 1e-5, 'momentum must accumulate across steps'